In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# FFT 기초: 수식 없이 주파수 분석 시작하기

진동 신호를 시간 영역에서 관찰하는 것만으로는 어떤 주기 성분이 들어 있는지 알기 어렵다. 이 노트북에서는 다음 두 단계로 주파수 분석의 기초 감각을 익힌다.

1. **실습 1.** 시간 영역의 신호를 불러오고, 일부 구간을 확대하여 주기성을 눈으로 확인한다.
2. **실습 2.** `utils.fft` 로 단측 진폭 스펙트럼을 구한다. 50 ~ 70 Hz 구간을 확대하여 빈(bin) 간격을 확인하고, 60 Hz 양옆의 작은 값이 무엇인지 판별한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils

---

## 실습 1. 신호 불러오기와 시간 영역 관찰

샘플링 주파수가 `fs = 1000 Hz` 이고 길이가 1초인 합성 진동 신호 `data_sample_fft.csv` 를 불러온다. 이 신호는 실측 데이터가 아니라 실습용으로 만든 것이다. 전체 구간(0 ~ 1 s)을 한 번 그린 뒤, 0 ~ 0.2 s 구간을 확대하여 반복되는 파형을 확인한다.

In [ ]:
fs = 1000

data = np.array(pd.read_csv('./data/data_sample_fft.csv'))

t = data[:, 1]
v = data[:, 2]

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Full record (0 ~ 1 s)')

ax[1].plot(t, v, 'C0')
ax[1].set_xlim([0, 0.2])
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Zoom-in (0 ~ 0.2 s)')

fig.tight_layout(); plt.show()

**관찰.** 전체 1초 구간에서는 값이 빠르게 변동하여 주기를 하나하나 재기 어렵다. 그러나 0 ~ 0.2 s 구간으로 확대하면 약 60 Hz 의 주기 성분이 뚜렷이 관찰되며, 그 주기는 1/60 s, 곧 약 16.7 ms 이다. 즉 신호는 무질서한 것이 아니며, 짧은 구간을 확대하면 구조가 나타난다. 다만 시간 영역의 관찰만으로는 성분의 개수와 진폭을 정량화하기 어렵고, 이것이 주파수 분석이 필요한 첫 번째 이유이다.

---

## 실습 2. `utils.fft` 로 단측 스펙트럼 그리기와 50 ~ 70 Hz 구간 확대

`utils.fft(v, fs)` 는 실수 신호에 대해 단측 진폭 스펙트럼 `(f, A)` 를 반환한다. 전체 스펙트럼은 `plot` 으로 나타내고, 60 Hz 주변의 50 ~ 70 Hz 구간은 `stem` 으로 확대하여 그린다. 확대한 그림은 세로축을 그대로 둔 것과 0 ~ 0.08로 좁힌 것의 두 가지이다. 이 그림들에서 빈 간격 `fs/N` 과 60 Hz 양옆 빈의 작은 값을 관찰한다.

In [ ]:
f, A = utils.fft(v, fs)
N = len(v)
print('bin spacing fs/N = ', fs / N, 'Hz')

fig, ax = plt.subplots(4, 1, figsize=(10, 12))

ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Time Domain')

# 포인트 수가 많을 때는 plot 이 가독성이 좋다
ax[1].plot(f, A, 'C1')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Frequency Domain (full)')

# 확대 구간은 개별 bin 이 보이도록 stem 이 적합
ax[2].stem(f, A, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[2].set_xlim([50, 70])
ax[2].set_xlabel('Frequency (Hz)'); ax[2].set_ylabel('Amplitude')
ax[2].set_title('Frequency Domain (zoom 50 ~ 70 Hz)')

# 같은 구간의 세로축을 0 ~ 0.08 로 좁혀 피크 양옆의 작은 값을 본다
ax[3].stem(f, A, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[3].set_xlim([50, 70]); ax[3].set_ylim([0, 0.08])
ax[3].set_xlabel('Frequency (Hz)'); ax[3].set_ylabel('Amplitude')
ax[3].set_title('Frequency Domain (zoom 50 ~ 70 Hz, amplitude 0 ~ 0.08)')

fig.tight_layout(); plt.show()
print('peak frequency :', f[np.argmax(A)], 'Hz')
print('peak amplitude :', A.max())

**관찰.** 마지막 두 줄은 `peak frequency : 60.0 Hz` 와 약 2.02의 `peak amplitude` 를 출력하며, 이는 신호에 포함된 60 Hz 사인파의 진폭이 약 2라는 뜻이다. 전체 스펙트럼에서는 60 Hz 의 피크 하나만 눈에 띄지만, 50 ~ 70 Hz 로 확대하면 빈 간격이 `fs/N = 1000/1000 = 1 Hz` 임을 확인할 수 있다. 세로축을 0 ~ 0.08로 좁혀 보면, 멀리 떨어진 54 Hz 의 값(약 0.062)이 60 Hz 바로 옆의 빈인 59 Hz(약 0.025)와 61 Hz(약 0.048)의 값보다 오히려 크다. 50 ~ 70 Hz 전체에도 비슷한 크기의 값이 고르게 나타난다. 이 분포는 피크에서 멀어질수록 값이 작아지는 누설의 모양이 아니므로, 이 작은 값들은 잡음이다. 측정 시간 1 s 동안 60 Hz 성분이 정확히 60번 진동하여 빈의 중심에 놓이기 때문에 이 신호에는 누설이 생기지 않는다. 누설은 신호의 주파수가 빈과 빈 사이에 놓일 때 발생한다.

---